In [ ]:
# ===================== 导入依赖库 =====================
import cv2
from skimage.util import random_noise
import numpy as np
from matplotlib import pyplot as plt
import os

# 自动创建保存结果的文件夹
if not os.path.exists('result'):
    os.makedirs('result')

## 1. 读取原始图像

In [ ]:
img = cv2.imread('cv--e2test.jpg')

# 获取像素点 (100, 100) 的 BGR 值
(b, g, r) = img[100, 100]
print('Pixel (100,100) BGR:', b, g, r)

plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title('Original Image (RGB View)')
plt.savefig('result/original_bgr.jpg', dpi=300)
plt.show()

## 2. 颜色空间转换

In [ ]:
# BGR → RGB
rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
plt.imshow(rgb_img)
plt.title('RGB Image')
plt.savefig('result/rgb_image.jpg', dpi=300)
plt.show()

# BGR → Gray
gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
plt.imshow(gray_img, cmap='gray')
plt.title('Gray Image')
plt.savefig('result/gray_image.jpg', dpi=300)
plt.show()

## 3. 添加噪声（椒盐噪声 & 高斯噪声）

In [ ]:
# 椒盐噪声
sp_noise_img = random_noise(rgb_img, mode='s&p', amount=0.4)

# 高斯噪声
gus_noise_img = random_noise(rgb_img, mode='gaussian', mean=0.2, var=0.03)

plt.figure(figsize=(10, 4))

plt.subplot(1, 3, 1)
plt.imshow(rgb_img)
plt.title('Original Image')

plt.subplot(1, 3, 2)
plt.imshow(sp_noise_img)
plt.title('S&P Noise')

plt.subplot(1, 3, 3)
plt.imshow(gus_noise_img)
plt.title('Gaussian Noise')

plt.tight_layout()
plt.savefig('result/noise_comparison.jpg', dpi=300)
plt.show()

## 4. 经典图像滤波方法对比（均值 / 中值 / 高斯 / 双边）

In [ ]:
# 转换数据格式以适应OpenCV滤波函数 (uint8, 0-255)
sp_uint8 = (sp_noise_img * 255).astype(np.uint8)
gus_uint8 = (gus_noise_img * 255).astype(np.uint8)

# 1. 均值滤波
mean_sp = cv2.blur(sp_uint8, (5, 5))
mean_gus = cv2.blur(gus_uint8, (5, 5))

# 2. 中值滤波
mid_sp = cv2.medianBlur(sp_uint8, 5)
mid_gus = cv2.medianBlur(gus_uint8, 5)

# 3. 高斯滤波
gauss_sp = cv2.GaussianBlur(sp_uint8, (5, 5), 0)
gauss_gus = cv2.GaussianBlur(gus_uint8, (5, 5), 0)

# 4. 双边滤波
bilat_sp = cv2.bilateralFilter(sp_uint8, 9, 75, 75)
bilat_gus = cv2.bilateralFilter(gus_uint8, 9, 75, 75)

In [ ]:
# 绘制四种滤波方法在椒盐噪声与高斯噪声下的对比图
plt.figure(figsize=(14, 8))

# 椒盐噪声组
plt.subplot(2, 4, 1)
plt.imshow(sp_uint8)
plt.title('S&P Noise')

plt.subplot(2, 4, 2)
plt.imshow(mean_sp)
plt.title('Mean Filter')

plt.subplot(2, 4, 3)
plt.imshow(mid_sp)
plt.title('Median Filter')

plt.subplot(2, 4, 4)
plt.imshow(bilat_sp)
plt.title('Bilateral Filter')

# 高斯噪声组
plt.subplot(2, 4, 5)
plt.imshow(gus_uint8)
plt.title('Gaussian Noise')

plt.subplot(2, 4, 6)
plt.imshow(mean_gus)
plt.title('Mean Filter')

plt.subplot(2, 4, 7)
plt.imshow(gauss_gus)
plt.title('Gaussian Filter')

plt.subplot(2, 4, 8)
plt.imshow(bilat_gus)
plt.title('Bilateral Filter')

plt.tight_layout()
plt.savefig('result/advanced_filter_comparison.jpg', dpi=300)
plt.show()

## 5. 滤波参数探索：不同卷积核大小对高斯滤波的影响

In [ ]:
# 分别测试 3x3, 5x5, 7x7 卷积核对高斯噪声的处理效果
k3_img = cv2.GaussianBlur(gus_uint8, (3, 3), 0)
k5_img = cv2.GaussianBlur(gus_uint8, (5, 5), 0)
k7_img = cv2.GaussianBlur(gus_uint8, (7, 7), 0)

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.imshow(k3_img)
plt.title('Kernel size: 3x3')

plt.subplot(1, 3, 2)
plt.imshow(k5_img)
plt.title('Kernel size: 5x5')

plt.subplot(1, 3, 3)
plt.imshow(k7_img)
plt.title('Kernel size: 7x7')

plt.tight_layout()
plt.savefig('result/kernel_size_comparison.jpg', dpi=300)
plt.show()

## 6. 手动实现彩色图像中值滤波

In [ ]:
def manual_median_filter_color(image, kernel_size=5):
    """
    手动实现彩色图像的中值滤波
    image: 输入彩色图像 (H, W, 3)
    kernel_size: 滤波窗口大小
    """
    pad = kernel_size // 2
    filtered_img = np.zeros_like(image)

    for c in range(3):
        channel = image[:, :, c]
        padded = np.pad(channel, pad_width=pad, mode='edge')
        for i in range(channel.shape[0]):
            for j in range(channel.shape[1]):
                region = padded[i:i+kernel_size, j:j+kernel_size]
                filtered_img[i, j, c] = np.median(region)

    return filtered_img

In [ ]:
manual_mid = manual_median_filter_color(sp_uint8, kernel_size=5)

plt.figure(figsize=(8, 4))

plt.subplot(1, 2, 1)
plt.imshow(sp_uint8)
plt.title('S&P Noise Image')

plt.subplot(1, 2, 2)
plt.imshow(manual_mid)
plt.title('Manual Median Filter')

plt.tight_layout()
plt.savefig('result/manual_median.jpg', dpi=300)
plt.show()